<a href="https://colab.research.google.com/github/hamzaarif727/ECHO---Quantathon/blob/main/ECHO_Hamza_Arif.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ECHO — Investigation Notebook
**Name:** Hamza Arif &nbsp;|&nbsp; **Roll No.:** CT-26082

**Event:** Entangled Minds: The Quantathon

**Mandatory modules:** 01, 03, 05 &nbsp;|&nbsp; **Stretch module:** 04 — The Erased Measurement

## Contents
0. Setup
1. Module 01 — The Gatekeeper
2. Module 03 — The Two Observers
3. Module 05 — The Damaged Message
4. Stretch Module 04 — The Erased Measurement

*Each module follows the same pattern: **Objective → Prediction (written before running) → Code → Result → What it proves**.*

## 0. Setup

In [44]:
!pip install -q qiskit qiskit-aer matplotlib pylatexenc

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
import numpy as np
import matplotlib.pyplot as plt
import random
from itertools import product

sim = AerSimulator()

def run(qc, shots=1000):
    """Run a circuit `shots` times; return counts like {'01': 480, '10': 520}."""
    return sim.run(qc, shots=shots).result().get_counts()

---
## 1. Module 01 — The Gatekeeper *(mandatory)*
**Objective:** Prove we can speak the machine's language: a bit that is always 1, a bit that is perfectly uncertain, a bit that looks uncertain but is certain, and the pattern `101`. Gates allowed: H, X, measure.

### Bronze — always-1 bit and 50/50 bit
**Prediction:** X on `|0>` gives `|1>`, so always `1`. H gives an equal blend, so about 500 `0` / 500 `1`.

In [45]:
k1 = QuantumCircuit(1, 1)
k1.x(0)
k1.measure(0, 0)    # X then measure

k2 = QuantumCircuit(1, 1)
k2.h(0)
k2.measure(0, 0)    # H then measure

c1, c2 = run(k1), run(k2)
print("always-1 bit: ", c1); print("50/50 bit   : ", c2)
plot_histogram([c1, c2], legend=["X", "H"]); plt.show()

always-1 bit:  {'1': 1000}
50/50 bit   :  {'1': 469, '0': 531}


**Result:** X always gives 1; H gives roughly 50/50.                   
**Proves:** X is a definite flip; H creates randomness only when we measure the blend.

### Silver — looks uncertain but is secretly certain
**Prediction:** H twice. After one H the qubit is a 50/50 blend, but the second H makes the two paths to `|1>` cancel (+0.5 and −0.5) while paths to `|0>` add, so the result is always `0`.

In [46]:
silver = QuantumCircuit(1, 1)
silver.h(0)
silver.h(0)
silver.measure(0, 0)       #after two H
print(run(silver))

s2 = QuantumCircuit(1)
s2.h(0)
print("\nafter one H :", np.round(Statevector(s2).data.real, 3))
s2.h(0)
print("after two H :", np.round(Statevector(s2).data.real, 3))

{'0': 1000}

after one H : [0.707 0.707]
after two H : [1. 0.]


**Result:** it is always `0`.     
**Proves:** *interference*: amplitudes (not probabilities) add, and they can cancel out each other. Two classical coin flips in a row would stay random, so this is genuinely quantum.

### Gold — build `101` from `000`
**Prediction:** flip qubits 0 and 2 with X, leave qubit 1 alone, so we always read `101`.

In [47]:
g = QuantumCircuit(3, 3)
g.x(0)
g.x(2)
g.measure([0, 1, 2], [0, 1, 2])
print(run(g))

print(g.draw())

{'101': 1000}
     ┌───┐   ┌─┐   
q_0: ┤ X ├───┤M├───
     └───┘┌─┐└╥┘   
q_1: ─────┤M├─╫────
     ┌───┐└╥┘ ║ ┌─┐
q_2: ┤ X ├─╫──╫─┤M├
     └───┘ ║  ║ └╥┘
c: 3/══════╩══╩══╩═
           1  0  2 


**Result:** `101` every time.   
**Proves:** gates act deterministically on definite states.

                                
**Module 01 summary:**    
*Answer:* always-1 bit, 50/50 bit, H·H = always 0, `101` built.      
*Code:* as mentioned above (bronze, silver and Gold).    
*What it proves:* randomness arises only at measurement of a superposition, and gates can cancel each other through interference.

---
## 2. Module 03 — The Two Observers *(mandatory, heart of the case)*
**Objective:** Zara's two detectors always agreed, and she called it proof of a link across space. Test whether "always agreeing" really shows that.

### Bronze — the pair that only ever agrees
**Concept:** H on qubit 0, then CNOT(0→1), gives an *entangled Bell pair*: `00` or `11` with amplitude 0.707 each. Neither qubit has a definite value alone, but the two always match.  
**Prediction:** only `00` and `11`, about 50% each; never `01` or `10`.

In [48]:
bell = QuantumCircuit(2, 2)
bell.h(0)
bell.cx(0, 1)

print("amplitudes (00,01,10,11):", np.round(Statevector(bell).data.real, 3))
bell.measure([0, 1], [0, 1])
bell_counts = run(bell)

print(bell.draw())
print(bell_counts)

plot_histogram(bell_counts)
plt.show()

amplitudes (00,01,10,11): [0.707 0.    0.    0.707]
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 
{'00': 510, '11': 490}


**Result:** agreeing pairs only `00`/`11`.  
**Proves:** the pair is perfectly correlated in this measurement.

### Silver — does "always agreeing" prove a quantum link?
**Prediction:** The clear Answer is No. Two people (assuming Alice and Bob) with a *pre-agreed secret coin flip* will always agree too, with nothing travelling between them. A simulation should look identical to the Bell result.

In [49]:
random.seed(1)
tally = {'00': 0, '11': 0, '01': 0, '10': 0}
for _ in range(1000):
    secret = random.choice('01')           # decided BEFORE they separate
    tally[secret + secret] += 1            # Alice and Bob both just read it
print(tally)

{'00': 485, '11': 515, '01': 0, '10': 0}


**Result:** same statistics as the quantum pair.  
**Proves:** agreement alone is *not* evidence of a quantum link or a message, rather a shared hidden secret explains it equally well.

**Extra check (supports the no-signalling point):** look at *one* detector on its own. Its result should be a random 50/50 even though the pair always agrees, so Alice's detector alone carries no message.

In [50]:
aliceOnly = {'0': 0, '1': 0}
for outcome, n in bell_counts.items():
    aliceOnly[outcome[-1]] += n          # rightmost character = qubit 0 (Alice)
print("Alice's detector on its own:", aliceOnly)

Alice's detector on its own: {'0': 510, '1': 490}


**Result:** Alice's result alone is about 50/50 random.   
**Proves:** the agreement only shows up when Alice's and Bob's lists are *compared*, and comparing needs an ordinary channel. So agreement is not a message.

### Gold — not attempted
I did not attempt Gold (the experiment that separates a real quantum pair from a pre-agreed secret). With only H, X and CNOT the two cannot be told apart; the standard test (CHSH, which needs a rotation gate) is the one I would run next, and I list it as my proposed next experiment in the report.   
So I **cannot** claim from my own results that the detectors' correlation is truly quantum or just a shared secret.

**Module 03 summary:**  
*Answer:* agreement alone does not prove a quantum link or a message.   
*What it proves:* the detectors *agreed*; nothing here shows they *sent* anything.    
*Code:* Reached Bronze and Silver.

---
## 3. Module 05 — The Damaged Message *(mandatory)*
**Objective:** each bit was sent three times and at most one of each triple was flipped (a *3-bit repetition code*). Recover the message.

### Bronze — recover the message
**Prediction:** trusting the value that appears at least twice (majority vote) fixes any single flip, giving 32 bits = 4 ASCII letters.

In [51]:
stream = '''100 011 001 001 001 011 100 110 001 011 000 000 001 100 011 011
000 111 000 000 110 000 001 000 000 111 001 010 101 011 111 111'''
triples = stream.split()
print("triples:", len(triples))

def majority(t):
    return '1' if t.count('1') >= 2 else '0'     # at least two 1s -> the bit was 1

bits = ''.join(majority(t) for t in triples)
message = ''.join(chr(int(bits[i:i+8], 2)) for i in range(0, 32, 8))   # 8 bits -> 1 letter
print("bits   :",  ' '.join(bits[i:i+8] for i in range(0, 32, 8)))
print("message:", message)
print("corrupted triples:", sum(t not in ('000', '111') for t in triples), "of", len(triples))

triples: 32
bits   : 01000101 01000011 01001000 01001111
message: ECHO
corrupted triples: 20 of 32


**Result:** the message is **ECHO**.  
**Proves:** at most one flip per triple is correctable by majority vote.

### Silver — where the trick fails
**Prediction:** with **two** flips in one triple, the majority points the wrong way (`000` → `110` decodes as `1`).

In [52]:
def send(bit, flips):
    t = [bit, bit, bit]
    for i in flips:
        t[i] = '1' if t[i] == '0' else '0'
    return ''.join(t)

for flips in [[], [0], [1], [2], [0, 1], [0, 1, 2]]:
    r = send('0', flips)
    print(f"sent 0, flips at {str(flips):10s} -> received {r} -> decoded {majority(r)}", "OK" if majority(r)=='0' else "WRONG")

sent 0, flips at []         -> received 000 -> decoded 0 OK
sent 0, flips at [0]        -> received 100 -> decoded 0 OK
sent 0, flips at [1]        -> received 010 -> decoded 0 OK
sent 0, flips at [2]        -> received 001 -> decoded 0 OK
sent 0, flips at [0, 1]     -> received 110 -> decoded 1 WRONG
sent 0, flips at [0, 1, 2]  -> received 111 -> decoded 1 WRONG


**Result:** fails exactly when 2+ flips occur in one triple.  
**Proves:** the code relies on the 'at most one flip' promise; `110` could be `111` with one flip or `000` with two, so the decoder cannot tell.

### Gold — (correct vs notice) why not copy a qubit?
* Codewords `000` and `111` differ in 3 places, so the code **corrects 1 flip** and **detects up to 2** (but would mis-correct them).
* **No-cloning:** an unknown quantum state cannot be copied. CNOT only copies definite values (`|0>`,`|1>`). On a superposition it creates **entanglement**:

In [53]:
q = QuantumCircuit(2)
q.h(0)
q.cx(0, 1)
print("after 'copying' |+> with CNOT:", np.round(Statevector(q).data.real, 3))
print("a true copy |+>|+> would be   : [0.5 0.5 0.5 0.5]")

after 'copying' |+> with CNOT: [0.707 0.    0.    0.707]
a true copy |+>|+> would be   : [0.5 0.5 0.5 0.5]


**Result:** entangled pair `(0.707, 0, 0, 0.707)`, not two copies.  
**Proves:** you can't protect a qubit by copying it three times; real quantum codes spread it across qubits with entanglement and check parities without measuring the data.

**Module 05 summary:**  
*Answer:* message = **ECHO**.   
*What it proves:*  Zara's last word is recoverable; the code corrects 1 flip per triple, and quantum states can't simply be copied.

---
## 4. Stretch Module 04 — The Erased Measurement
**Objective:** a log entry "changed when it was observed". ARIA blames the investigators. Is that physics or a cover story?

### Bronze — recoverable vs destroyed
**Prediction:** `H, H` is recoverable (no information lost): always `0`. `H, measure, H, measure` is destroyed: the mid-circuit measurement collapses the blend, so the second result is random.

In [54]:
rec = QuantumCircuit(1, 1)
rec.h(0)
rec.h(0)
rec.measure(0, 0)

des = QuantumCircuit(1, 2)
des.h(0)
des.measure(0, 0)      # mid-circuit measurement: collapses the blend
des.h(0)
des.measure(0, 1)      # second measurement (bitstring is printed 'second first')
print("recoverable H,H         : ", run(rec))
print("destroyed H,measure,H   : ", run(des))

recoverable H,H         :  {'0': 1000}
destroyed H,measure,H   :  {'01': 252, '10': 251, '00': 245, '11': 252}


**Result:** recoverable is always `0`; destroyed gives all four combinations about equally.   
**Proves:** observing a superposition changes it for good. 'Changed when observed' is real physics, and by itself is not evidence of tampering.

### Silver — predict first, then run
**Prediction:** `X,H,H` → H·H cancels so net effect is X → always `1`. `H,X,H` equals the Z gate, which only flips the sign of the `|1>` amplitude; starting from `|0>` nothing visible changes → always `0`.

In [55]:
def build(seq):
    qc = QuantumCircuit(1, 1)
    for g in seq:
        qc.x(0) if g == 'X' else qc.h(0)
    qc.measure(0, 0)
    return qc
for seq in ['XHH', 'HXH']:
    print(seq, run(build(seq)))

XHH {'1': 1000}
HXH {'0': 1000}


**Result:** matches. **Proves:** the apparent 'erasing' is interference (amplitudes cancelling), not data being deleted.
A log entry that looks changed doesn’t have to mean someone tampered with it. It can be an ordinary effect of gates and measurement.

### Gold — predict any sequence before running
Apply gates left to right to the amplitude pair (a0, a1): **X** swaps them; **H** maps to ((a0+a1)/√2, (a0−a1)/√2). Probability = amplitude².

In [56]:
def predict(seq):
    a0, a1 = 1.0, 0.0                      # start in |0>
    for g in seq:
        if g == 'X': a0, a1 = a1, a0
        else:        a0, a1 = (a0 + a1)/np.sqrt(2), (a0 - a1)/np.sqrt(2)
    return float(round(a0**2, 3)), float(round(a1**2, 3))

for seq in ['XHH', 'HXH', 'HXHX', 'HHXH', 'XHXH']:
    print(f"{seq:6s} predicted (P0,P1) = {predict(seq)}   simulated = {run(build(seq), 2000)}")

XHH    predicted (P0,P1) = (0.0, 1.0)   simulated = {'1': 2000}
HXH    predicted (P0,P1) = (1.0, 0.0)   simulated = {'0': 2000}
HXHX   predicted (P0,P1) = (0.0, 1.0)   simulated = {'1': 2000}
HHXH   predicted (P0,P1) = (0.5, 0.5)   simulated = {'0': 988, '1': 1012}
XHXH   predicted (P0,P1) = (0.0, 1.0)   simulated = {'1': 2000}


**Result:** predictions match the simulation. **Proves:** the behaviour is fully predictable from the gate rules.

**Module 04 summary:**  
*Answer:* the 'erased' entry is explained by ordinary physics (measurement collapse and interference).  
*What it proves:* ARIA's excuse is *physically possible*, but it cannot show ARIA is honest or that nobody tampered with the log.